# Tutorial 2 — WorldView-3 UCSD

The commercial-stereo recipe. Two WV3 panchromatic NTFs to an orthorectified, ICESat-2-aligned 2 m DEM.

We chose this dataset because:

- Openly available. SpaceNet CORE3D imagery is on a public AWS bucket with `--no-sign-request`. No Vantor license, no API keys.
- Well-conditioned geometry: 21° convergence, 0.37 base-to-height, 12-day temporal separation.
- Mixed landscape: mesa-top neighborhoods cut by brushy canyons east of UCSD.
- The ROI sits on an ICESat-2 ground-track crossover, so the alignment step has dense validation points.

## Pair details

| Catid | Date | Off-nadir | Position |
|---|---|---|---|
| `1040010007A93700` | 2015-02-12 | 8.4° | left |
| `1040010007CA4D00` | 2015-02-24 | 12.9° | right |

ROI: 3 × 3 km east of the UCSD campus. UTM zone 11N (`EPSG:32611`).

## What we'll do

1. Download both NTFs and metadata from SpaceNet S3.
2. Visualize the stereo geometry (skyplot, footprints).
3. Clip a Copernicus DEM tile from AWS Open Data.
4. Run `mapproject` to orthorectify both images onto the COP-DEM grid, cropped to the ROI (ASP calls orthorectification "mapprojection").
5. Run stereo and DEM generation.
6. Generate a full report including ICESat-2 alignment via `pc_align`.


In [ ]:
from pathlib import Path

DATA = "../data/ucsd_stereo_21deg_12d"

!mkdir -p {DATA}

# Configuration shared across cells
T_PROJWIN = "477750 3634800 480750 3637800"                               # 3x3 km ROI: mesas + canyons east of UCSD
TR = 0.5                                                                  # orthorectification GSD; coarser than the native 0.315 m to speed processing
REF_DEM = f"{DATA}/ref/cop30_ucsd.tif"


## 1. Download the imagery from SpaceNet

Each WV3 scene comes as an NTF (the imagery) plus a TAR (metadata, including the IMD with RPC coefficients and an XML camera). The download script handles the AWS fetch + tar extract and renames files to short, human-readable names (`<CATID>_P001.NTF`, `<CATID>_P001.xml`).

In [ ]:
!bash ../scripts/download_worldview_ucsd.sh {DATA}
!ls -lh {DATA}/*.NTF {DATA}/*.xml

## 2. Stereo geometry

`StereoGeometryPlotter` reads the XML metadata and builds a skyplot (where each satellite was looking, in azimuth/elevation) plus a map view of the two image footprints.


In [ ]:
from asp_plot.stereo_geometry import StereoGeometryPlotter

sgp = StereoGeometryPlotter(DATA)
print(f"UTM EPSG: {sgp.get_pair_utm_epsg()}")
print(f"Scene bounds (lon/lat): {sgp.get_scene_bounds()}")
sgp.dg_geom_plot()

## 3. Reference DEM from AWS Copernicus DEM

Orthorectification needs a coarse reference DEM. Copernicus GLO-30 is on AWS Open Data; no API key, no auth.

`scene_bbox` runs ASP's `camera_footprint` on each image and unions the results; `utm_epsg` picks the UTM zone at the bbox center. `fetch_cop_dem.py` then clips, mosaics, and reprojects the matching COP30 tiles.


In [ ]:
from utils import scene_bbox, utm_epsg

BBOX = scene_bbox(
    (f"{DATA}/1040010007A93700_P001.NTF", f"{DATA}/1040010007A93700_P001.xml"),
    (f"{DATA}/1040010007CA4D00_P001.NTF", f"{DATA}/1040010007CA4D00_P001.xml"),
)
T_SRS = utm_epsg(BBOX)
print(f"Scene bbox: {BBOX}  UTM: {T_SRS}")

Path(f"{DATA}/ref").mkdir(exist_ok=True)

if Path(REF_DEM).exists():
    print(f"{REF_DEM} exists — skipping COP30 fetch. Delete it to redownload.")
else:
    !python ../scripts/fetch_cop_dem.py \
        --bbox {BBOX} \
        --t-srs {T_SRS} \
        --tr 30 \
        --out {REF_DEM}

!ls -lh {REF_DEM}


## 4. Mapproject

Resample both NTFs onto the COP-DEM grid at 0.5 m, coarser than the native 0.315 m GSD, to speed processing. The vendor RPC cameras are used as delivered; `--t_projwin` crops to the 3 × 3 km ROI.


In [ ]:
%%time
for catid in ["1040010007A93700", "1040010007CA4D00"]:
    out = f"{DATA}/{catid}_P001_map.tif"
    if Path(out).exists():
        print(f"{out} exists — skipping mapproject for {catid}. Delete it to reprocess.")
    else:
        !mapproject --threads 4 \
            --t_srs {T_SRS} --tr {TR} \
            --t_projwin {T_PROJWIN} \
            {REF_DEM} \
            {DATA}/{catid}_P001.NTF \
            {DATA}/{catid}_P001.xml \
            {out}


## 5. Stereo + DEM

Stereo on the orthorectified pair. The inputs are now nearly aligned in geographic space, so the matcher's search range is small. Pass the reference DEM as the final argument so the matcher knows the grid.

`point2dem` grids the point cloud at 2 m, roughly 4× the image GSD, a common rule of thumb for stereo DEM resolution.

Thread budget tuned for the 4-core Codespace floor: `asp_mgm` is memory-intensive and multithreads well, so ASP recommends few processes with many threads — here 1 process with 4 threads. On a larger machine, scale `--processes` to your core count divided by 4.


In [ ]:
%%time
if Path(f"{DATA}/stereo/run-DEM.tif").exists():
    print(f"{DATA}/stereo/run-DEM.tif exists — skipping stereo + point2dem. Delete {DATA}/stereo/ to reprocess.")
else:
    !parallel_stereo \
        --processes 1 \
        --threads-multiprocess 4 \
        --threads-singleprocess 4 \
        --stereo-algorithm asp_mgm \
        --subpixel-mode 9 \
        {DATA}/1040010007A93700_P001_map.tif \
        {DATA}/1040010007CA4D00_P001_map.tif \
        {DATA}/1040010007A93700_P001.xml \
        {DATA}/1040010007CA4D00_P001.xml \
        {DATA}/stereo/run \
        {REF_DEM}

    !point2dem --tr 2.0 -r earth --errorimage {DATA}/stereo/run-PC.tif

!ls -lh {DATA}/stereo/run-DEM.tif


## 6. Full report with ICESat-2 alignment

The `asp_plot` CLI produces a PDF report: input scenes, disparity, hillshades, dh vs. the reference DEM, and an ICESat-2 ATL06-SR comparison. By default it also runs `pc_align` to register the DEM to ICESat-2 and appends the alignment report.

Expect this cell to take a few minutes; it fetches ATL06-SR data via SlideRule and runs `pc_align`.


In [ ]:
%%time
if Path(f"{DATA}/stereo/ucsd_wv3_report.pdf").exists():
    print(f"{DATA}/stereo/ucsd_wv3_report.pdf exists — skipping asp_plot CLI. Delete it to regenerate.")
else:
    !asp_plot \
        --directory {DATA} \
        --stereo_directory stereo \
        --reference_dem {DATA}/ref/cop30_ucsd.tif \
        --subset_km 0.5 \
        --report_filename ucsd_wv3_report.pdf


The report is at: [data/ucsd_stereo_21deg_12d/stereo/ucsd_wv3_report.pdf](../data/ucsd_stereo_21deg_12d/stereo/ucsd_wv3_report.pdf).

## What's next

→ To see what bundle adjustment adds, run [`03_worldview_ucsd_ba.ipynb`](03_worldview_ucsd_ba.ipynb) and compare the reports.

→ For more advanced WV processing (jitter correction with `jitter_solve`, no-orthorectification variants, multi-pair scene selection), see [`asp_plot`'s WorldView notebooks](https://asp-plot.readthedocs.io/en/latest/examples/index.html).

→ For planetary stereo (Mars CTX/HiRISE/MOC, Lunar LRO NAC), see [`asp_plot`'s planetary notebooks](https://asp-plot.readthedocs.io/en/latest/examples/index.html). The recipe is the same; the sensor-prep step changes.
